# 03 - Data Preparation

## Tujuan
Mengubah teks mentah menjadi representasi numerik siap model: case folding,
pembersihan, deteksi bahasa, pembentukan n-gram, stemming Sastrawi, serta
penyaringan stopword.

## Asal (Input)
`data/processed/thesis_for_pipeline.csv` - 309 dokumen hasil Notebook 02

## Keluaran (Output)
- `data/intermediate/dataset_preprocessed.csv` - 285 dokumen + kolom tokens
- `data/intermediate/dictionary.gensim` - 1.849 kata unik
- `data/intermediate/corpus.pkl` - corpus Bag-of-Words

## Tujuh Tahap Prapemrosesan
1. Case Folding | 2. Cleaning | 3. Deteksi Bahasa & Fallback
4. Bigram & Trigram (Gensim Phrases) | 5. Stemming (Sastrawi)
6. Filter Stopword Post-Stemming | 7. Filter Token Pendek

In [1]:
import sys, io as _io
from pathlib import Path
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_ROOT))
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import pandas as pd, numpy as np
from src import config as C
C.ensure_dirs()
print(C.describe())

KONFIGURASI PROYEK
  Repo          : C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main
  Dokumen akhir : 285 (2021, 2022, 2023, 2024, 2025)
  Vocabulary    : 1849
  Model LDA     : K=8, alpha=asymmetric, eta=auto, passes=30, iterations=500, seed=42
  Inferensi     : Online Variational Bayes (bukan Gibbs Sampling)
  C_V resmi     : 0.4256
  Cutoff data   : 19 September 2026

## Langkah 1 - Jalankan Prapemrosesan 7 Tahap

In [2]:
from src import preprocessing as pp

pp.run_preprocessing(use_tfidf=False,
                     min_ngram_count=C.MIN_NGRAM_COUNT,
                     output_dir=str(C.INTERMEDIATE_DIR))

2026-10-12 15:33:17,701 [INFO] ============================================================
2026-10-12 15:33:17,701 [INFO] PREPROCESSING PIPELINE
2026-10-12 15:33:17,712 [INFO] ============================================================
2026-10-12 15:33:17,712 [INFO] 
[1/5] Memuat data...
2026-10-12 15:33:17,734 [INFO]   Total Dokumen mentah: 309
2026-10-12 15:33:17,742 [INFO]   Total Dokumen setelah hapus duplikat judul: 305 (dibuang: 4)
2026-10-12 15:33:17,742 [INFO]   Dengan abstrak: 302
2026-10-12 15:33:17,742 [INFO] 
[2/5] Deteksi & Filter Teks (Abstrak Bahasa Inggris)...
2026-10-12 15:33:17,844 [INFO]   Dokumen dengan Abstrak Bahasa Inggris difilter (fallback ke judul): 57
2026-10-12 15:33:17,909 [INFO] 
[3/5] Membentuk Model N-Gram (Bigram & Trigram)...
2026-10-12 15:33:17,909 [INFO] collecting all words and their counts
2026-10-12 15:33:17,909 [INFO] PROGRESS: at sentence #0, processed 0 words and 0 word types
2026-10-12 15:33:17,962 [INFO] collected 22446 token types (unigram

## Langkah 2 - Verifikasi Hasil Prapemrosesan

In [3]:
prep = pd.read_csv(C.PREPROCESSED_CSV)
print(f'Dokumen : {len(prep)}')
print(f'Tahun   : {prep.groupby("Tahun").size().to_dict()}')
print()
avg = prep['tokens'].apply(lambda x: len(eval(x))).mean()
print(f'Rata-rata token per dokumen: {avg:.1f}')

Dokumen : 285
Tahun   : {2021: 29, 2022: 78, 2023: 46, 2024: 47, 2025: 85}

Rata-rata token per dokumen: 58.1

## Langkah 3 - Contoh Hasil Tokenisasi

In [4]:
import ast
sample = prep.head(3)
for _, r in sample.iterrows():
    toks = ast.literal_eval(r['tokens'])
    print(f'  [{int(r["Tahun"])}] {str(r["Judul"])[:55]}')
    print(f'         {toks[:12]} ...')
    print()

  [2021] SISTEM INFORMASI LAYANAN ASURANSI BERBASIS\r\nANDROID D
         ['layan_asuransi', 'android', 'service', 'ajb_bumiputera', 'cabang', 'sekip', 'rangka_tingkat', 'kualitas', 'usaha', 'ajb_bumiputera', 'cabang', 'sekip'] ...

  [2021] PEMANFAATAN HUMAN ORGANIZATION TECHNOLOGY-NET \r\nBENEF
         ['human_organization_technology_net', 'benefit', 'ticketing', 'kereta_api', 'indonesia', 'persero', 'divre', 'iii', 'kertapati', 'kereta_api', 'indonesia', 'kai'] ...

  [2021] SISTEM INFORMASI PERSATUAN AMAL KEMATIAN DESA \r\nSUNGA
         ['amal', 'mati', 'desa', 'sungai', 'pinang', 'camat', 'rambutan', 'kabupaten_banyuasin'] ...

## Langkah 4 - Verifikasi Dictionary

In [5]:
from gensim import corpora
d = corpora.Dictionary.load(str(C.DICTIONARY_PATH))
print(f'Vocabulary : {len(d)} kata/n-gram')
print(f'Nilai exp  : {C.VOCAB_SIZE}')
print('Status     :', 'SESHUI' if len(d) == C.VOCAB_SIZE else 'BERBEDA')

Vocabulary : 1849 kata/n-gram
Nilai exp  : 1849
Status     : SESHUI

## Ringkasan Fase 03 - Data Preparation

| Metrik | Nilai |
|:---|---:|
| Dokumen output | 285 |
| Ukuran vocabulary | 1.849 |
| Rata-rata token/dokumen | 58,1 |

Prapemrosesan ketat ini menghasilkan rata-rata hanya 58 token per dokumen.
Kosakata yang terbatas ini menurunkan C_V karena konteks ko-okurensi kata
menyempit - hal yang dibahas sebagai keterbatasan pada Bab IV skripsi.

## Peta Artefak

| Artefak | Dibuat oleh | Consumed by |
|:---|:---|:---|
| `data/intermediate/dataset_preprocessed.csv` | Notebook 03 | Notebook 04, 04b, 05 |
| `data/intermediate/dictionary.gensim` | Notebook 03 | Notebook 04, 04b, 08 |
| `data/intermediate/corpus.pkl` | Notebook 03 | Notebook 04, 05, 08 |